In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

url = "https://codis.cwa.gov.tw/api/station?"
headers = {
    "accept": "application/json, text/javascript, */*; q=0.01",
    "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
    "x-requested-with": "XMLHttpRequest",
}

mountain_df = pd.read_csv('csv/mountain_two_simple.csv')
station_ids = set()
station_ids.update(mountain_df['站號'].dropna())
station_ids.update(mountain_df['站號2'].dropna())

start_date = datetime(2016, 4, 29)
end_date = datetime(2025, 8, 14)

for stn_id in station_ids:
    all_data = []
    current_date = start_date
    
    while current_date < end_date:
        next_date = min(current_date + timedelta(days=30), end_date)
        
        data = {
            "date": current_date.strftime("%Y-%m-%dT%H:%M:%S.000+08:00"),
            "type": "report_date",
            "stn_ID": stn_id,
            "stn_type": "auto_C0",
            "more": "",
            "start": current_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "end": next_date.strftime("%Y-%m-%dT23:59:59"),
            "item": "",
        }
        
        response = requests.post(url, headers=headers, data=data, verify=False)
        result = response.json()
        
        if 'data' in result:
            all_data.extend(result['data'])
        
        current_date = next_date + timedelta(days=1)
    
    if all_data:
        df = pd.DataFrame(all_data)
        df.to_csv(f'weather_csv/{stn_id}_weather.csv', index=False)